In [ ]:
import databento as db
import matplotlib.pyplot as plt

# Set parameters
equity_datasets = [
        "XNAS.ITCH",  # Nasdaq
        "XBOS.ITCH",  # Nasdaq BX
        "XPSX.ITCH",  # Nasdaq PSX
        "XNYS.PILLAR",  # NYSE
        "ARCX.PILLAR",  # NYSE Arca
        "XASE.PILLAR",  # NYSE American
        "XCHI.PILLAR",  # NYSE Texas
        "XCIS.TRADESBBO",  # NYSE National
        "MEMX.MEMOIR",  # Members Exchange
        "EPRL.DOM",  # MIAX Pearl
        "IEXG.TOPS",  # IEX
        "BATS.PITCH",  # Cboe BZX
        "BATY.PITCH",  # Cboe BYX
        "EDGA.PITCH",  # Cboe EDGA
        "EDGX.PITCH",  # Cboe EDGX
    ]
symbols = ["QQQ"]
start = "2025-12-19"
end = "2025-12-20"

client = db.Historical("db-UGgrjRPHWKJAvF9taFUYD6TnBbirC")

# Request OHLCV-1d data for the selected symbols
df = client.timeseries.get_range(
    dataset=dataset,
    symbols=symbols,
    schema="ohlcv-1d",
    start=start,
    end=end,
).to_df()



In [ ]:
equity_datasets = [
    "XNAS.ITCH", "XBOS.ITCH", "XPSX.ITCH", "XNYS.PILLAR", "ARCX.PILLAR",
    "XASE.PILLAR", "XCHI.PILLAR", "XCIS.TRADESBBO", "MEMX.MEMOIR", "EPRL.DOM",
    "IEXG.TOPS", "BATS.PITCH", "BATY.PITCH", "EDGA.PITCH", "EDGX.PITCH",
]

df_all2 = pd.DataFrame()

symbols = ["QQQ"]
start = "2025-12-16" #pd.Timestamp("2025-12-19 09:30", tz="US/Eastern")
end   = "2025-12-19" #pd.Timestamp("2025-12-19 16:00", tz="US/Eastern")

client = db.Historical("db-UGgrjRPHWKJAvF9taFUYD6TnBbirC")

dfs = []
for ds in equity_datasets:
    try:
        tmp = client.timeseries.get_range(
            dataset=ds,
            symbols=symbols,
            schema="ohlcv-1m",
            start=start,
            end=end,
        ).to_df()

        tmp["equity_dataset"] = ds
        dfs.append(tmp)

    except Exception as e:
        # some venue feeds may not support ohlcv-1m; skip and continue
        print(f"Skipping {ds}: {e}")

df_all2 = pd.concat(dfs, ignore_index=False).reset_index()
df_all2['volume'].sum()

In [ ]:
df_main = df_comb.reset_index()[['symbol', 'ts_event', 'close', 'open', 'high', 'low', 'volume', 'equity_dataset']].copy()

# Add in session duration to account for early close on holidays
df_main['datetime_est'] = (df_main['ts_event'].dt.tz_convert('America/New_York'))
df_close_times = close_times()
# Merge close times with intraday data
df_main['Date'] = pd.to_datetime(df_main['datetime_est']).dt.strftime('%Y-%m-%d')
df_close_times["Date"] = pd.to_datetime(df_close_times["Date"]).dt.date
df_main["Date"] = pd.to_datetime(df_main["Date"]).dt.date
df_main = df_main.merge(df_close_times[['Date', 'session_duration']], on="Date", how="left")

df_main

In [ ]:
df_main.sort_values(by='datetime_est')
from datetime import time

df_cons = (
    df_main
    .loc[
        df_labeled_final["datetime_est"].dt.time.between(
            time(9, 30), time(15, 59)
        )
    ]
    .groupby("datetime_est", as_index=False)
    .agg(
        volume=("volume", "sum"),
        low=("low", "min"),
        high=("high", "max"),
        close=("close", "mean"),
        open=("open", "mean"),
    )
    .assign(
        open=lambda x: x["open"].round(2),
        close=lambda x: x["close"].round(2),
    )
    .sort_values("datetime_est")
)
df_cons.to_csv('db.csv')

In [ ]:
df_intraday_labels = add_intraday_labels(df_main)
df_intraday_labels['Date'] = pd.to_datetime(df_intraday_labels['datetime_est']).dt.strftime('%Y-%m-%d')
df_labeled_final = df_intraday_labels[['symbol', 'datetime_est', 'time_to_open', 'time_to_close', 'session_simple', 
                    'session_detail', 'close', 'open', 'high', 'low', 'Date', 'session_duration', 'volume', 'equity_dataset']].copy()

In [ ]:
volume_by_date_ds = (
    df_labeled_final
    .loc[df_labeled_final["session_simple"] == "open_market"]
    .assign(Date=lambda x: x["datetime_est"].dt.date)
    .groupby(["Date", "equity_dataset"], as_index=False)
    .agg(total_volume=("volume", "sum"))
    .sort_values(["Date", "equity_dataset"])
)
volume_by_date_ds

In [ ]:
df_orig = yf.Ticker('QQQ').history(start='2025-12-16', end='2025-12-20', interval="1m", auto_adjust=True).reset_index()[['Datetime', 'Close', 'High', 'Low', 'Volume']]
df_orig['Date'] = pd.to_datetime(df_orig['Datetime']).dt.strftime('%Y-%m-%d')
df_orig.to_csv('yf.csv')